In [0]:
USE CATALOG esg_assurance;
USE SCHEMA adi_assurance;

In [0]:
SELECT current_catalog(), current_schema();

In [0]:
CREATE OR REPLACE TABLE assurance_workpaper_plan
(
    workpaper_id STRING,
    metric_scope STRING,
    assertion STRING,
    risk_statement STRING,
    control_id STRING,
    control_description STRING,
    test_procedure STRING,
    evidence_required STRING,
    testing_method STRING,
    result STRING,
    conclusion STRING
);

In [0]:
INSERT INTO assurance_workpaper_plan VALUES

(
    'WP-001',
    'Scope 3 Cat. 1 and Cat. 2',
    'Occurrence / Accuracy',
    'Duplicate procurement records could overstate Scope 3 emissions.',
    'CTRL-DQ-001',
    'Detect duplicate transaction identifiers.',
    'Identify duplicate transaction IDs, inspect physical records, verify supporting evidence, confirm approved remediation, and retest the population.',
    'Source transaction records; record hashes; remediation approval; retest result.',
    'Inspection + Reperformance',
    NULL,
    NULL
),

(
    'WP-002',
    'Scope 3 Cat. 1 and Cat. 2',
    'Completeness / Traceability',
    'Missing supplier identifiers could prevent transaction-level traceability.',
    'CTRL-DQ-002',
    'Validate supplier identifiers against required source fields.',
    'Identify transactions with missing supplier IDs and determine whether the supplier can be supported by source evidence.',
    'Procurement record; supplier master; supporting invoice or supplier evidence.',
    'Inspection',
    NULL,
    NULL
),

(
    'WP-003',
    'Scope 3 Cat. 1 and Cat. 2',
    'Accuracy / Master Data',
    'Incorrect supplier geography could result in inappropriate supplier or emission-factor treatment.',
    'CTRL-DQ-003',
    'Compare transaction supplier country with supplier master.',
    'Reperform the country comparison and investigate mismatches.',
    'Supplier master; transaction record; supporting supplier documentation.',
    'Reperformance',
    NULL,
    NULL
),

(
    'WP-004',
    'Scope 3 Cat. 1 and Cat. 2',
    'Cutoff',
    'Transactions outside FY2025 could be included in the reporting population.',
    'CTRL-DQ-004',
    'Validate transaction date against reporting-period boundaries.',
    'Inspect transactions outside the defined period and verify whether inclusion is appropriate.',
    'Invoice date; reporting-period definition; accounting record.',
    'Inspection + Reperformance',
    NULL,
    NULL
),

(
    'WP-005',
    'Scope 3 Cat. 1 and Cat. 2',
    'Classification',
    'Transactions could be assigned to an incorrect Scope 3 category.',
    'CTRL-DQ-005',
    'Compare reported category with controlled category mapping.',
    'Reperform category mapping from source classification to Scope 3 category.',
    'Category mapping table; source GL/service classification; methodology.',
    'Reperformance',
    NULL,
    NULL
),

(
    'WP-006',
    'Scope 3 Cat. 1 and Cat. 2',
    'Accuracy',
    'Incorrect FX conversion could distort spend-based emissions.',
    'CTRL-DQ-007',
    'Recalculate USD spend from original currency amount and FX rate.',
    'Reperform FX calculation and investigate differences outside tolerance.',
    'Original spend; currency; FX rate; calculated USD spend.',
    'Reperformance',
    NULL,
    NULL
),

(
    'WP-007',
    'Scope 3 Cat. 1 and Cat. 2',
    'Accuracy',
    'Reported emissions may not be reproducible from activity data and emission factors.',
    'CTRL-DQ-008',
    'Reperform transaction-level emissions calculation.',
    'Independently recalculate emissions using the stored activity data and approved factor.',
    'Activity data; emission factor; methodology; calculation output.',
    'Reperformance',
    NULL,
    NULL
),

(
    'WP-008',
    'Scope 3 Cat. 1 and Cat. 2',
    'Methodology / Change Control',
    'Retired or unapproved emission factors could be used.',
    'CTRL-DQ-009',
    'Validate factor approval status and effective period.',
    'Inspect emission-factor version, status, and applicability for the reporting period.',
    'Emission factor master; version metadata; approval status.',
    'Inspection',
    NULL,
    NULL
),

(
    'WP-009',
    'Scope 3 Cat. 1 and Cat. 2',
    'Accuracy',
    'Emission-factor units may be inconsistent with activity-data units.',
    'CTRL-DQ-010',
    'Compare transaction factor unit with controlled factor-master unit.',
    'Inspect unit consistency and reperform calculation where mismatches exist.',
    'Factor master; transaction factor unit; methodology.',
    'Inspection + Reperformance',
    NULL,
    NULL
),

(
    'WP-010',
    'Supplier-specific Scope 3',
    'Evidence',
    'Supplier-specific calculations may lack sufficient supporting evidence.',
    'CTRL-DQ-011',
    'Require evidence reference for supplier-specific calculations.',
    'Inspect supplier-specific records and trace each to supporting evidence.',
    'Supplier disclosure; evidence register; calculation record.',
    'Inspection',
    NULL,
    NULL
),

(
    'WP-011',
    'Scope 3 Cat. 1 and Cat. 2',
    'Authorization / Governance',
    'Manual overrides could change reported emissions without appropriate review.',
    'CTRL-DQ-012',
    'Require documented rationale and approval for manual overrides.',
    'Inspect override records, rationale, reviewer status, and approval evidence.',
    'Override log; reviewer approval; supporting rationale.',
    'Inspection',
    NULL,
    NULL
),

(
    'WP-012',
    'Gold Scope 3 disclosure',
    'Completeness / Accuracy',
    'Gold disclosure totals may not reconcile to the eligible Silver population.',
    'CTRL-REC-001',
    'Reconcile Gold totals to reporting-eligible Silver records.',
    'Independently sum eligible Silver emissions and compare with Gold disclosure output.',
    'Silver eligibility table; Gold category summary; reconciliation result.',
    'Reperformance',
    NULL,
    NULL
);

In [0]:
SELECT
    workpaper_id,
    assertion,
    control_id,
    testing_method,
    risk_statement
FROM assurance_workpaper_plan
ORDER BY workpaper_id;

In [0]:
CREATE OR REPLACE VIEW assurance_control_status AS

SELECT
    control_id,

    CASE
        WHEN SUM(
            CASE
                WHEN remediation_status <> 'CLOSED'
                THEN 1 ELSE 0
            END
        ) > 0
        THEN 'FAIL'

        ELSE 'PASS'
    END AS control_result,

    COUNT(*) AS total_exceptions,

    SUM(
        CASE
            WHEN remediation_status <> 'CLOSED'
            THEN 1 ELSE 0
        END
    ) AS open_exceptions,

    SUM(
        CASE
            WHEN remediation_status = 'CLOSED'
            THEN 1 ELSE 0
        END
    ) AS closed_exceptions

FROM assurance_exception_log

GROUP BY control_id;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_control_status
ORDER BY control_id;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_additional_control_results AS

SELECT
    'CTRL-REC-001' AS control_id,
    'PASS' AS control_result,
    0 AS total_exceptions,
    0 AS open_exceptions,
    0 AS closed_exceptions;

In [0]:
CREATE OR REPLACE VIEW
esg_assurance.adi_assurance.assurance_all_control_status AS

SELECT *
FROM esg_assurance.adi_assurance.assurance_control_status

UNION ALL

SELECT *
FROM esg_assurance.adi_assurance.assurance_additional_control_results;

In [0]:
MERGE INTO esg_assurance.adi_assurance.assurance_workpaper_plan w

USING esg_assurance.adi_assurance.assurance_all_control_status c

ON w.control_id = c.control_id

WHEN MATCHED THEN
UPDATE SET
    w.result = c.control_result;

In [0]:
SELECT
    workpaper_id,
    control_id,
    assertion,
    result
FROM esg_assurance.adi_assurance.assurance_workpaper_plan
ORDER BY workpaper_id;

 Step 8.2 is now working exactly as intended.
Result shows:
- WP-001 / CTRL-DQ-001 = PASS
- WP-012 / CTRL-REC-001 = PASS
- the remaining 10 workpapers = FAIL because unresolved exceptions still exist
That is the right pattern. It proves the framework is distinguishing between:
- a remediated/retested duplicate control,
- a successful source-to-disclosure reconciliation,
- and unresolved control issues still blocking full support.

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_findings_register AS

SELECT
    w.workpaper_id,
    w.control_id,
    w.assertion,
    w.risk_statement,
    w.result,

    COUNT(DISTINCT e.transaction_id) AS affected_transactions,

    ROUND(
        SUM(e.emissions_exposure_tco2e),
        6
    ) AS gross_exception_exposure_tco2e,

    MAX(e.initial_severity) AS initial_severity,

    CASE
        WHEN COUNT(DISTINCT e.transaction_id) = 0
        THEN 'NO_OPEN_FINDING'

        WHEN MAX(e.initial_severity) = 'CRITICAL'
        THEN 'HIGH_PRIORITY'

        WHEN MAX(e.initial_severity) = 'HIGH'
        THEN 'PRIORITY'

        ELSE 'STANDARD'
    END AS finding_priority,

    CASE
        WHEN COUNT(DISTINCT e.transaction_id) > 0
        THEN 'OPEN'
        ELSE 'CLOSED'
    END AS finding_status,

    CASE
        WHEN COUNT(DISTINCT e.transaction_id) > 0
        THEN 'Additional evidence or remediation is required before this assertion is fully supported.'
        ELSE 'No unresolved exception identified.'
    END AS assurance_conclusion

FROM esg_assurance.adi_assurance.assurance_workpaper_plan w

LEFT JOIN esg_assurance.adi_assurance.assurance_exception_log e
    ON w.control_id = e.control_id
   AND e.remediation_status <> 'CLOSED'

GROUP BY
    w.workpaper_id,
    w.control_id,
    w.assertion,
    w.risk_statement,
    w.result;

In [0]:
SELECT
    workpaper_id,
    control_id,
    assertion,
    affected_transactions,
    gross_exception_exposure_tco2e,
    initial_severity,
    finding_priority,
    finding_status
FROM esg_assurance.adi_assurance.assurance_findings_register
ORDER BY workpaper_id;

In [0]:
SELECT
    finding_status,
    finding_priority,
    COUNT(*) AS findings
FROM esg_assurance.adi_assurance.assurance_findings_register
GROUP BY
    finding_status,
    finding_priority
ORDER BY
    finding_status,
    finding_priority;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_pillar_mapping
(
    control_id STRING,
    kpmg_pillar STRING,
    rationale STRING
);

In [0]:
INSERT INTO esg_assurance.adi_assurance.assurance_pillar_mapping VALUES

('CTRL-DQ-001',
 'Data Management',
 'Duplicate detection supports accuracy, completeness, and controlled data processing.'),

('CTRL-DQ-002',
 'Value Chain',
 'Supplier identification is necessary for traceability across the upstream value chain.'),

('CTRL-DQ-003',
 'Value Chain',
 'Supplier master-data consistency supports supplier-level traceability and correct geography.'),

('CTRL-DQ-004',
 'Data Management',
 'Reporting-period cutoff controls support completeness and period accuracy.'),

('CTRL-DQ-005',
 'Data Management',
 'Controlled category mapping supports consistent Scope 3 classification.'),

('CTRL-DQ-007',
 'Data Management',
 'FX reperformance supports accuracy of spend-based calculations.'),

('CTRL-DQ-008',
 'Data Management',
 'Independent recalculation supports reproducibility of emissions results.'),

('CTRL-DQ-009',
 'Governance',
 'Emission-factor approval and version control support methodology governance and change control.'),

('CTRL-DQ-010',
 'Data Management',
 'Unit consistency supports calculation accuracy and methodological integrity.'),

('CTRL-DQ-011',
 'Value Chain',
 'Supplier-specific calculations require traceable evidence from value-chain partners.'),

('CTRL-DQ-012',
 'Governance',
 'Manual overrides require authorization, rationale, and documented review.'),

('CTRL-REC-001',
 'Digital Technology',
 'Automated reconciliation between Silver and Gold demonstrates system-enabled traceability and reproducibility.');

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_pillar_mapping
ORDER BY kpmg_pillar, control_id;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_readiness_summary AS

SELECT
    m.kpmg_pillar,

    COUNT(*) AS controls_tested,

    SUM(
        CASE
            WHEN w.result = 'PASS'
            THEN 1 ELSE 0
        END
    ) AS controls_passed,

    SUM(
        CASE
            WHEN w.result = 'FAIL'
            THEN 1 ELSE 0
        END
    ) AS controls_failed,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN w.result = 'PASS'
                THEN 1 ELSE 0
            END
        )
        / COUNT(*),
        1
    ) AS tested_control_pass_rate_pct,

    CASE
        WHEN SUM(
            CASE
                WHEN w.result = 'FAIL'
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'NO_OPEN_FINDINGS'

        WHEN SUM(
            CASE
                WHEN w.result = 'FAIL'
                THEN 1 ELSE 0
            END
        ) <= 1
        THEN 'LIMITED_OPEN_FINDINGS'

        ELSE 'MULTIPLE_OPEN_FINDINGS'
    END AS readiness_observation

FROM esg_assurance.adi_assurance.assurance_pillar_mapping m

JOIN esg_assurance.adi_assurance.assurance_workpaper_plan w
    ON m.control_id = w.control_id

GROUP BY m.kpmg_pillar;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_readiness_summary
ORDER BY kpmg_pillar;

This tested_control_pass_rate_pct is not a KPMG maturity score.
It means only:
“Among the controls we chose to test in this synthetic case, what proportion currently pass?”

That distinction matters because KPMG’s actual Index is based on a broader survey methodology and weighted pillars, not on this control-testing exercise.

In [0]:
SELECT
    kpmg_pillar,
    controls_tested,
    controls_passed,
    controls_failed,
    tested_control_pass_rate_pct,
    readiness_observation
FROM esg_assurance.adi_assurance.assurance_readiness_summary
ORDER BY kpmg_pillar;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_readiness_narrative AS

SELECT
    kpmg_pillar,

    CASE

        WHEN kpmg_pillar = 'Governance'
        THEN
        'Open findings remain around emission-factor approval and manual override authorization. Governance controls require further remediation before broader assurance readiness.'

        WHEN kpmg_pillar = 'Data Management'
        THEN
        'The pipeline demonstrates automated validation and successful Gold reconciliation, but unresolved cutoff, classification, FX, recalculation, and unit-control exceptions remain.'

        WHEN kpmg_pillar = 'Digital Technology'
        THEN
        'The Databricks pipeline supports traceable Bronze-Silver-Gold processing and successful source-to-disclosure reconciliation.'

        WHEN kpmg_pillar = 'Value Chain'
        THEN
        'Supplier identification, geography, and supplier-specific evidence exceptions remain open, indicating value-chain data quality and evidence gaps.'

        ELSE
        'Not assessed.'

    END AS readiness_narrative

FROM esg_assurance.adi_assurance.assurance_readiness_summary;

In [0]:
INSERT INTO esg_assurance.adi_assurance.assurance_readiness_narrative
VALUES
(
    'Skills',
    'Not assessed in this synthetic data-control exercise. No conclusion is drawn regarding staff competency, training, or ESG assurance expertise.'
);

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_readiness_narrative
ORDER BY kpmg_pillar;